In [2]:
#Utilty functions
from Bio import SeqIO

def clean_loci(loci):
    """
    Cleans and processes loci of interest by removing introns, appending upstream and downstream sequences, and ensuring uppercase format.

    Arguments:
        loci (str) - input loci DNA sequence
        
    Returns:
        loci - cleaned loci sequence 
        loci_exon - exon sequence extracted from loci (assuming introns are coded as lower case)     
    """
    #Remove any leading or trailing blank spaces from loci
    loci = loci.strip()
    
    #Extract exons 
    loci_exon = ""
    for base in loci:
        if base.isupper():
            loci_exon += base

    return loci, loci_exon

def reverse_complement(dna_seq):
    """ 
    Return the reverse complement of a DNA sequence. 
    Arguments:
        dna_seq - DNA input sequence as a string
        
    Returns:
        reverse_complement - Reverse complement of DNA input sequence 
    
    """
    complement = {'a': 't', 'A':'T', 't': 'a', 'T':'A','c': 'g','C':'G', 'g': 'c', 'G':'C'}
    
    reverse_complement_seq = ''.join(complement.get(base, base) for base in reversed(dna_seq))
        
    return reverse_complement_seq

def translate(dna_seq):
    """
    Generates a protein sequence from a DNA sequence inputted as a string. It will only convert if the exon is in upper
    case

    Arguments:
        dna_seq(str) - input DNA sequence
        
    Returns:
        protein_sequence(str) - output protein sequence 
    """
    
    #Generate dictionary of codons for dna to protein translation
    
    codon_table = {
    'GCT': 'A', 'GCC': 'A', 'GCA': 'A', 'GCG': 'A',  # Alanine
    'TGT': 'C', 'TGC': 'C',                          # Cysteine
    'GAT': 'D', 'GAC': 'D',                          # Aspartic acid
    'GAA': 'E', 'GAG': 'E',                          # Glutamic acid
    'TTT': 'F', 'TTC': 'F',                          # Phenylalanine
    'GGT': 'G', 'GGC': 'G', 'GGA': 'G', 'GGG': 'G',  # Glycine
    'CAT': 'H', 'CAC': 'H',                          # Histidine
    'ATT': 'I', 'ATC': 'I', 'ATA': 'I',              # Isoleucine
    'AAA': 'K', 'AAG': 'K',                          # Lysine
    'TTA': 'L', 'TTG': 'L', 'CTT': 'L', 'CTC': 'L', 'CTA': 'L', 'CTG': 'L',  # Leucine
    'ATG': 'M',                                      # Methionine (Start codon)
    'AAT': 'N', 'AAC': 'N',                          # Asparagine
    'CCT': 'P', 'CCC': 'P', 'CCA': 'P', 'CCG': 'P',  # Proline
    'CAA': 'Q', 'CAG': 'Q',                          # Glutamine
    'CGT': 'R', 'CGC': 'R', 'CGA': 'R', 'CGG': 'R', 'AGA': 'R', 'AGG': 'R',  # Arginine
    'TCT': 'S', 'TCC': 'S', 'TCA': 'S', 'TCG': 'S', 'AGT': 'S', 'AGC': 'S',  # Serine
    'ACT': 'T', 'ACC': 'T', 'ACA': 'T', 'ACG': 'T',  # Threonine
    'GTT': 'V', 'GTC': 'V', 'GTA': 'V', 'GTG': 'V',  # Valine
    'TGG': 'W',                                      # Tryptophan
    'TAT': 'Y', 'TAC': 'Y',                          # Tyrosine
    'TAA': '*', 'TAG': '*', 'TGA': '*'               # Stop codons
    }
    
    # Initialize the protein sequence
    protein = ""

    # Iterate over the DNA sequence in steps of 3 (codon length)
    for i in range(0, len(dna_seq) - len(dna_seq) % 3, 3):
        
        # Define a codon as 3 bases
        codon = dna_seq[i:i + 3]
        # Translate the codon into an amino acid based on the dictionary
        if codon.isupper():
            protein += codon_table.get(codon, '?')  # '?' is a placeholder for unknown codons
        else:
            protein += 'x'

    return protein

def extract_genome_multifast_to_list_of_sequence_records(genome_multi_fasta):
    '''
    Takes in genomic multi fasta input and list of sequence records out 

    Args:
    genome_multi_fasta(fasta): multi fasta containing sequence records for a genome

    Returns:
    genome_sequence_records(list): list of sequences in biopython seqRecord format
    '''
    print('Starting genome extraction...')
    genome_sequence_records = []
    for record in SeqIO.parse(genome_multi_fasta, "fasta"):
        genome_sequence_records.append(record)
    print('Converted to list of sequence records')
    
    return genome_sequence_records

In [3]:
#Tests for utils

In [4]:
#loci extraction
import pandas as pd
import requests
from app_utils import translate, reverse_complement, clean_loci, extract_genome_multifast_to_list_of_sequence_records

'''
This script will take in uniprot IDs and extract the genomic loci in format (introns in lowercase and exons in uppercase). This format is used by downstream processes to design guide RNAs,
homology directed repair templates and amplicon specific primers.
'''
def extract_genomic_information_from_uniprot_id(uniprot_id):
    '''
    Takes in uniprot ID as a string input and pings the Uniprot API to extract genomic coordinates of the protein and exons.
    Metadata such as name, taxID, protein sequence, genome assembly name,  ENSEMBL GeneID, ENSEMBL Transcript ID and ENSEMBL Translations IDs is included alongside the extracted coordinates.  

    Args:
    uniprot_id (str): uniprot ID

    Returns:
    genomic_information (pd.DataFrame): DataFrame containing genomic coordinates of the protein of interest alongside exon positions and metadata 
    '''
    genomic_information = pd.DataFrame()
    try:
        print(f'Searching for UniProt ID: {uniprot_id}')
        requestURL_protein = f"https://www.ebi.ac.uk/proteins/api/coordinates/{uniprot_id}"
        response_protein = requests.get(requestURL_protein, headers={"Accept": "application/json"})
        
        # Check if the request was successful
        response_protein.raise_for_status()
        
        # Load JSON response
        response_protein = response_protein.json()
        
        # Check if response is not empty
        if response_protein:
            response_protein_normalise = pd.json_normalize(
                response_protein, 
                record_path=['gnCoordinate', 'genomicLocation', 'exon'], 
                meta=['accession', 'name', 'taxid', 'sequence', 
                      ['gnCoordinate', 'genomicLocation', 'chromosome'], 
                      ['gnCoordinate', 'genomicLocation', 'start'], 
                      ['gnCoordinate', 'genomicLocation', 'end'], 
                      ['gnCoordinate', 'genomicLocation', 'reverseStrand'], 
                      ['gnCoordinate', 'genomicLocation', 'nucleotideId'], 
                      ['gnCoordinate', 'genomicLocation', 'assemblyName'], 
                      ['gnCoordinate', 'ensemblGeneId'], 
                      ['gnCoordinate', 'ensemblTranscriptId'], 
                      ['gnCoordinate', 'ensemblTranslationId']],
                record_prefix='exon_'
            )

            # Group and aggregate exon information
            response_protein_normalise = response_protein_normalise.groupby([
                'accession', 'name', 'taxid', 'sequence', 
                'gnCoordinate.genomicLocation.chromosome', 
                'gnCoordinate.genomicLocation.start', 
                'gnCoordinate.genomicLocation.end', 
                'gnCoordinate.genomicLocation.reverseStrand', 
                'gnCoordinate.genomicLocation.nucleotideId', 
                'gnCoordinate.genomicLocation.assemblyName', 
                'gnCoordinate.ensemblGeneId', 
                'gnCoordinate.ensemblTranscriptId', 
                'gnCoordinate.ensemblTranslationId'
            ]).agg({
                'exon_id': lambda x: ','.join(map(str, x)),
                'exon_proteinLocation.begin.position': lambda x: ','.join(map(str, x)),                    
                'exon_proteinLocation.end.position': lambda x: ','.join(map(str, x)),
                'exon_genomeLocation.begin.position': lambda x: ','.join(map(str, x)),                    
                'exon_genomeLocation.end.position': lambda x: ','.join(map(str, x))
            }).reset_index()

            # Concatenate to the main DataFrame
            genomic_information = pd.concat([genomic_information, response_protein_normalise], ignore_index=True)
        else:
            print(f"No data found for UniProt ID: {uniprot_id}")
            
    except Exception as e:
        print(f"An error occurred: {e}")

    return genomic_information

def extract_genomic_loci_from_genomic_information(genomic_information, genome_sequence_records, UTR_length=500):
    '''                  
    Takes the output from the Uniprot API from extract_genomic_information_from_uniprot_id function and extracts the genomic loci using a given reference genome. It appends the genomic loci, 
    genomic loci exons and performs a translation check between the protein sequence meta data outputted from extract_genomic_information_from_uniprot_id and the translation of the genomic loci exons
    outputted from this function 

    Args:
    genomic_information (pd.DataFrame): dataframe input from the output of extract_genomic_information_from_uniprot_id

    Returns:
    genomic_information_loci_extracted (pd.DataFrame): dataframe output with extracted loci, loci_exons and translation checks
    '''
    
    chromosome_id = genomic_information['gnCoordinate.genomicLocation.chromosome'].iloc[0]
    records_dict = {seq.id: seq for seq in genome_sequence_records}
    match = records_dict.get(chromosome_id)
    
    if match:
        is_reverse_strand = genomic_information['gnCoordinate.genomicLocation.reverseStrand'].iloc[0] == 'True'
        uniprot_start = int(genomic_information['gnCoordinate.genomicLocation.start'].iloc[0])
        uniprot_end = int(genomic_information['gnCoordinate.genomicLocation.end'].iloc[0])
        
        # Adjust start and end for reverse strand
        if is_reverse_strand:
            uniprot_start, uniprot_end = uniprot_end, uniprot_start
        
        extracted_gene = match.seq[uniprot_start - UTR_length - 1:uniprot_end + UTR_length].lower()
        extracted_gene = str(extracted_gene)
        
        # Preparing exons DataFrame by splitting on commas
        df_exons_id = genomic_information[['exon_id']].copy()
        df_exons_id['exon_id'] = df_exons_id['exon_id'].str.split(',')
        
        df_exons_start = genomic_information[['exon_genomeLocation.begin.position']].copy()
        df_exons_start['exon_genomeLocation.begin.position'] = df_exons_start['exon_genomeLocation.begin.position'].str.split(',')
        
        df_exons_end = genomic_information[['exon_genomeLocation.end.position']].copy()
        df_exons_end['exon_genomeLocation.end.position'] = df_exons_end['exon_genomeLocation.end.position'].str.split(',')
        
        # Exploding columns
        df_exons_id = df_exons_id.explode('exon_id')
        df_exons_start = df_exons_start.explode('exon_genomeLocation.begin.position')
        df_exons_end = df_exons_end.explode('exon_genomeLocation.end.position')
        
        df_exons = pd.concat([df_exons_id, df_exons_start, df_exons_end], axis=1)
        df_exons.columns = ['exon_id', 'exon_start', 'exon_end']
        
        if is_reverse_strand:
            df_exons['exon_start'], df_exons['exon_end'] = df_exons['exon_end'], df_exons['exon_start']
        
        # Convert each exon sequence to uppercase in the extracted gene sequence
        for _, row in df_exons.iterrows():
            exon_start = int(row['exon_start'])
            exon_end = int(row['exon_end'])
            exon = str(match.seq[exon_start - 1:exon_end].lower())
            exon_start_index = extracted_gene.find(exon)
            
            if exon_start_index != -1:
                exon_end_index = exon_start_index + len(exon)
                extracted_gene = (
                    extracted_gene[:exon_start_index]
                    + extracted_gene[exon_start_index:exon_end_index].upper()
                    + extracted_gene[exon_end_index:]
                )
        
        if is_reverse_strand:
            annotated_gene = reverse_complement(extracted_gene)
        else:
            annotated_gene = extracted_gene
        
        genomic_loci, genomic_loci_exons = clean_loci(annotated_gene)
        genomic_information_loci_extracted = genomic_information.assign(genomic_loci=genomic_loci, genomic_loci_exon=genomic_loci_exons)
        
        # Translate the extracted loci exons and match them against the uniprot sequence 
        translation = translate(genomic_loci_exons)
        if translation == genomic_information_loci_extracted['sequence'].iloc[0]:
            genomic_information_loci_extracted = genomic_information_loci_extracted.assign(translation_check='Pass')
        else:
            genomic_information_loci_extracted = genomic_information_loci_extracted.assign(translation_check='Fail')
    
    return genomic_information_loci_extracted

In [5]:
# tests for loci extraction
test_multipart_fasta = r'C:\Users\dasak\OneDrive\Documents\Imperial\Foundry_Child_Lab\CRISPR-TAPE-2\code\reference_genomes\Homo_sapiens.GRCh38.dna.primary_assembly.fa'
test_sequence_records = extract_genome_multifast_to_list_of_sequence_records(test_multipart_fasta)
print(test_sequence_records)
test_uniprot_id = 'A0A075B6Q4'
test_genomic_information = extract_genomic_information_from_uniprot_id(test_uniprot_id)
display(test_genomic_information)
test_genomic_loci_extracted = extract_genomic_loci_from_genomic_information(test_genomic_information, test_sequence_records)
display(test_genomic_loci_extracted)

Starting genome extraction...
Converted to list of sequence records
[SeqRecord(seq=Seq('NNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNN...NNN'), id='1', name='1', description='1 dna:chromosome chromosome:GRCh38:1:1:248956422:1 REF', dbxrefs=[]), SeqRecord(seq=Seq('NNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNN...NNN'), id='10', name='10', description='10 dna:chromosome chromosome:GRCh38:10:1:133797422:1 REF', dbxrefs=[]), SeqRecord(seq=Seq('NNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNN...NNN'), id='11', name='11', description='11 dna:chromosome chromosome:GRCh38:11:1:135086622:1 REF', dbxrefs=[]), SeqRecord(seq=Seq('NNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNN...NNN'), id='12', name='12', description='12 dna:chromosome chromosome:GRCh38:12:1:133275309:1 REF', dbxrefs=[]), SeqRecord(seq=Seq('NNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNN...NNN'), id='13', name='13', description='13 dna:chromosome chromosome:GRCh38:13:1:114364328:1 REF', dbxrefs=

,accession,name,taxid,sequence,gnCoordinate.genomicLocation.chromosome,gnCoordinate.genomicLocation.start,gnCoordinate.genomicLocation.end,gnCoordinate.genomicLocation.reverseStrand,gnCoordinate.genomicLocation.nucleotideId,gnCoordinate.genomicLocation.assemblyName,gnCoordinate.ensemblGeneId,gnCoordinate.ensemblTranscriptId,gnCoordinate.ensemblTranslationId,exon_id,exon_proteinLocation.begin.position,exon_proteinLocation.end.position,exon_genomeLocation.begin.position,exon_genomeLocation.end.position
0,A0A075B6Q4,A0A075B6Q4_HUMAN,9606,MDIQKSENEDDSEWEDVDDEKGDSNDDYDSAGLLSDEDCMSVPGKT...,6,143857434,143884328,False,CM000668.2,GRCh38,ENSG00000280148,ENST00000454207,ENSP00000400756,"ENSE00003758190,ENSE00003759740,ENSE0000375999...","1,4,90,132","4,89,132,140","143857434,143857752,143860426,143884304","143857444,143858007,143860553,143884328"


,accession,name,taxid,sequence,gnCoordinate.genomicLocation.chromosome,gnCoordinate.genomicLocation.start,gnCoordinate.genomicLocation.end,gnCoordinate.genomicLocation.reverseStrand,gnCoordinate.genomicLocation.nucleotideId,gnCoordinate.genomicLocation.assemblyName,...,gnCoordinate.ensemblTranscriptId,gnCoordinate.ensemblTranslationId,exon_id,exon_proteinLocation.begin.position,exon_proteinLocation.end.position,exon_genomeLocation.begin.position,exon_genomeLocation.end.position,genomic_loci,genomic_loci_exon,translation_check
0,A0A075B6Q4,A0A075B6Q4_HUMAN,9606,MDIQKSENEDDSEWEDVDDEKGDSNDDYDSAGLLSDEDCMSVPGKT...,6,143857434,143884328,False,CM000668.2,GRCh38,...,ENST00000454207,ENSP00000400756,"ENSE00003758190,ENSE00003759740,ENSE0000375999...","1,4,90,132","4,89,132,140","143857434,143857752,143860426,143884304","143857444,143858007,143860553,143884328",gtctgtcccttagcagagcttgagcactgtgccgggagatccgctg...,ATGGATATACAGAAATCTGAGAATGAAGATGACAGCGAGTGGGAAG...,Pass


In [6]:
# guide generation for specific amino acid
def find_guides(loci, pam):
    """ 
    Finds guide RNA sequences on the positive and negative strands of the loci by 
    searching for PAM sites and extracting 20 bases upstream from the PAM position. 
    
    Arguments:
        loci - DNA input sequence as a string
        pam - protospacer adjacent motif for Cas protein
        
    Returns:
        pam_start_position - position of pam start site (adjusted for python 0 indexing)
        crRNA_DNA_sequence - sequence 20bp upstream of PAM
        strand - which strand PAM was found on (positive or negative)
    """
    
    # Initialize lists to store gRNA information
    pam_found = []
    pam_start_position = []
    crRNA_DNA_sequence = []
    strand = []
    
    # Define PAM motifs
    if pam == "NGG":
        print(f'Searching forward strand for {pam} sites...')
        print(loci)
        for n in range(len(loci) - 2):  # Ensure n + 2 in the following line is within bounds
            if (loci[n] in ['A','a','T','t','C','c','G','g'] and loci[n+1] in ['G','g'] and loci[n+2] in ['G','g']):
                if n - 21 >= 0:  # Ensure there are enough bases before the PAM
                    pam_found.append(loci[n:n+3])
                    pam_start_position.append(n) # pam start site
                    crRNA_DNA_sequence.append(loci[n-21:n])  # Extract sequence (adjusted for python 0 indexing and python slicing being non inclusive of the final value)
                    strand.append("forward")
        
        print(f'Searching reverse strand for {pam} sites...')
        loci_reverse_complement = reverse_complement(loci)
        print(loci_reverse_complement)
        for n in range(len(loci_reverse_complement) - 2):  # Ensure n + 2 in the following line is within bounds
            if (loci_reverse_complement[n] in ['A','a','T','t','C','c','G','g'] and loci_reverse_complement[n+1] in ['G','g'] and loci_reverse_complement[n+2] in ['G','g']):
                if n - 21 >= 0:  # Ensure there are enough bases before the PAM
                    pam_found.append(loci_reverse_complement[n:n+3])
                    pam_start_position.append(len(loci) - (n))  #pam start site
                    crRNA_DNA_sequence.append(loci_reverse_complement[n-21:n])  # Extract sequence
                    strand.append("reverse")
                    
    else:
        raise ValueError("PAM not recognised")

    return pam_found, pam_start_position, crRNA_DNA_sequence, strand

#get codon index function
def get_codon_index(loci, loci_exon):
    """ 
    Maps the positions of the protein coding DNA sequences (exons) along the loci. It gives information 
    regarding the amino acid residue in each position.

    Args:
    loci (str): extracted DNA loci sequence 
    loci_exon (str): extracted DNA loci exon sequence

    Returns:
    protein_dict (dict): a dictionary where each base in the loci exon is mapped to a protein and its location in the loci
    """
    assert len(loci_exon) % 3 == 0, "The length of the coding sequence is not a multiple of 3" #Check that exon is a multiple of 3
    Base_df = pd.DataFrame(list(loci), columns = ["Base"]) #List of all the bases in the genomic loci
    Base_df["Position"] = Base_df.index
    Base_df = Base_df[Base_df['Base'].str.isupper()].reset_index(drop=True) # Only
    pos1 = Base_df[Base_df.index % 3 == 0].reset_index(drop=True) #Generate a list of every third base from the first base
    pos2 = Base_df[Base_df.index % 3 == 1].reset_index(drop=True) #Generate a list of every third base from the second base
    pos3 = Base_df[Base_df.index % 3 == 2].reset_index(drop=True) #Generate a list of every third base from the third base
    aas = list(translate(loci_exon))
    protein_dict = {}
    for aa in range(len(aas)):
        protein_dict[aa] = {"Amino Acid": aas[aa],
                        "base_1": (pos1["Position"][aa],
                                pos1["Base"][aa]),
                        "base_2": (pos2["Position"][aa],
                                pos2["Base"][aa]),
                        "base_3": (pos3["Position"][aa],
                                pos3["Base"][aa])}

    return protein_dict

def get_GC_content(grna_dna_sequence):
    
    """ 
    Return the G/C content of the guide RNA sequence as a percentage.

    args: 
    grna_dna_sequence (str) : dna complement of gRNA variable region

    returns:
    gc_percentage (float): %GC of the dna complement of the gRNA variable region

    """
    gc_count = 0
    base_count = 0
    for char in grna_dna_sequence:
        if char.isalpha(): #Count the length of the guide RNA
            gc_count += 1
        if char in ['C','c','G','g']:
            base_count += 1 #Count the number of Gs or Cs in the guide RNA
    gc_percentage = float(base_count)/float(gc_count) * 100 #Calculate a percentage of Gs and Cs in the length of the guide RNA
    gc_percentage = round(gc_percentage, 2) #Round to two decimal places
    return gc_percentage

def guide_RNA_notes(grna_dna_sequence, gc_percentage):
    """ 
    Return additional information on the genereated guide RNA sequences.
    args:
    grna_dna_sequence (str) : dna complement of gRNA variable region
    gc_percentage (float): %GC of the dna complement of the gRNA variable region
    
    returns:
    notes (list): list of additional information for the guide RNA sequence 

    """

    # Check if guides have been identified
    if grna_dna_sequence == "No 5' guide could be identified" or grna_dna_sequence == "No 3' guide could be identified":
        return ""
    
    # Initialise the notes  
    notes = []
    for n in range(len(grna_dna_sequence)-3):
        if grna_dna_sequence[n:n + 4] == 'TTTT': #Check for four thymines in a row
            notes.append('PolyT present.')
    if grna_dna_sequence[0] != 'G': #Check the guide RNA starts with a 'G' at the most 5' position
        notes.append('No leading G.')
    if gc_percentage >= 75:
        notes.append('G/C content over 75%.') #Check if the G/C content of the guide is more than or equal to 75%
    return notes

def specific_function_guide_RNA_generation(spec_aa_pos, loci, reference_genome, minimum_distance = 30, maximum_distance = 60, pam = 'NGG'):
    """ 
    Identifies and produces metrics for guides upstream and downstream from a specific amino acid residue.
    
    Arguments:
        spec_aa_pos - specific amino acid residue selected by position
        minimum_distance - minimum search distance to find guide RNA (in bases pairs from the codon for the specific amino acid)
        minimum_distance - maximum search distance to find guide RNA (in bases pairs from the codon for the specific amino acid)
        pam - pam sequence e.g. NGG for spCas9
        loci - DNA loci for protein of interest 
        reference genome - reference genome (used for for off target metrics)
        
        
    Returns:
        guides (pd.Dataframe): pandas dataframe containing information about all guide pairs
    
    """
  
    loci, loci_exon = clean_loci(loci)
    print(f'loci (length:{len(loci)})')
    print(loci)
    print(f'loci exon (length:{len(loci_exon)})')
    print(loci_exon)
    
    # convert the CDS to a protein dictionary
    protein_dict = get_codon_index(loci, loci_exon)
    
    # display information about the selected amino acid
    spec_aa_pos = int(spec_aa_pos)
    maximum_distance = int(maximum_distance)
    aa_selected = protein_dict[spec_aa_pos - 1]
    # Confirm the selection is the correct amino acid
    print(f'The amino acid you have selected is {aa_selected} at aa position {spec_aa_pos} and loci postion {protein_dict[spec_aa_pos - 1]["base_1"][0]} - {protein_dict[spec_aa_pos - 1]["base_3"][0]}')  
    print(f'The PAM you have selected is {pam}')
    
    # generate lists of all the potential guides within the genomic loci and a list of all guides on the reverse complement.
    loci_region_start = protein_dict[spec_aa_pos - 1]["base_1"][0] - 2*maximum_distance
    loci_region_end = protein_dict[spec_aa_pos - 1]["base_3"][0] + 2*maximum_distance
    loci_region = loci[loci_region_start:loci_region_end]
    print(f'Searching for gRNAs in loci region {loci_region_start}-{loci_region_end}...')
    print(loci_region)

    
    # Identify PAMs in the inputted genomic loci
    pam_found , guide_positions, gRNA_list, guide_strands = find_guides(loci_region, pam)  
    
    # convert the lists into dataframes
    guide_df = pd.DataFrame()
    
    # Add a column for the pam
    guide_df["PAM"] = pam_found
    
    # Add a column of guide RNA sequences to the dataframe
    guide_df["Full gRNA Sequence"] = gRNA_list  
    
    # Add a column of guide RNA cut site positions to the dataframe
    guide_df["Position on + strand"] = guide_positions + loci_region_start
                        
    # Add a column to strand relative to the inputted genomic loci
    guide_df["Strand"] = guide_strands
    
    # Sort the guide RNAs by their position in the genomic loci
    guide_df = guide_df.sort_values(by=["Position on + strand"]).reset_index(drop=True)
    
    # Stops code if guide dataframe is empty
    assert not guide_df.empty, "No guide RNAs identified in sequence"
    
    print(f'{(len(guide_df))} guides found')
    
   # get the distance of the gRNAs from the start and end base of the amino acid
    start_base = protein_dict[spec_aa_pos - 1]["base_1"][0] #correct for python 0 index
    end_base = protein_dict[spec_aa_pos - 1]["base_3"][0] #correct for python 0 index
    guide_df["Distance from start of Amino Acid (bp)"] = guide_df["Position on + strand"] - start_base
    guide_df["Distance from end of Amino Acid (bp)"] = guide_df["Position on + strand"] - end_base
    
    print('guide_df')

    # define a dataframe for if there are no guides within the distance range
    noguides = pd.DataFrame(columns=["PAM","Full gRNA Sequence","Position on + strand","Strand","Distance from Amino Acid (bp)"])  
       
    # get the upstream guide RNA dataframe
    upstream_guides = []
    upstream_guides = guide_df.query(f"`Distance from start of Amino Acid (bp)` <= {-minimum_distance}").query(f"`Distance from start of Amino Acid (bp)` >= {-maximum_distance}")
    upstream_guides = upstream_guides.rename(columns={"Distance from start of Amino Acid (bp)": "Distance from Amino Acid (bp)"})
    upstream_guides = upstream_guides.drop("Distance from end of Amino Acid (bp)", axis=1)
    print('upstream_guides')
    print(upstream_guides)
    
    if len(upstream_guides) == 0:
        upstream_guides = noguides
    
    # get the downstream guide RNA dataframe
    downstream_guides = []
    downstream_guides = guide_df.query(f"`Distance from end of Amino Acid (bp)` >= {minimum_distance}").query(f"`Distance from end of Amino Acid (bp)` <= {maximum_distance}")
    downstream_guides = downstream_guides.rename(columns={"Distance from end of Amino Acid (bp)": "Distance from Amino Acid (bp)"})
    downstream_guides = downstream_guides.drop("Distance from start of Amino Acid (bp)", axis=1)
    print('downstream_guides')
    print(downstream_guides)
    
    if len(downstream_guides) == 0:
        downstream_guides = noguides
    
    # join the dataframes together
    guides = pd.concat([upstream_guides,downstream_guides])
    guides = guides.reset_index(drop = True)
    
    # get the G/C content
    guides["G/C Content (%)"] = guides.apply(
        lambda row: (
            get_GC_content(row["Full gRNA Sequence"])
            if row["Full gRNA Sequence"] != ""
            else ""
        ),
        axis=1
    )
    # get the notes
    guides["Notes"] = guides.apply(
        lambda row: (
            guide_RNA_notes(row["Full gRNA Sequence"], row["G/C Content (%)"])
            if row["Full gRNA Sequence"] != ""
            else ""
        ),
        axis=1
    )  
    
    # Perform off target searches
    guides["off target count (loci)"] = guides.apply(
    lambda row: loci.count(row['Full gRNA Sequence'] + row['PAM']) + 
                 loci.count(reverse_complement(row['Full gRNA Sequence'] + row['PAM'])) - 1,  
    axis=1
)
    guides["off target count (genome)"] = guides.apply(
        lambda row: sum(
            record.seq.count((row['Full gRNA Sequence'] + row['PAM']).upper()) +
            record.seq.count(reverse_complement((row['Full gRNA Sequence'] + row['PAM'] ).upper()))
            for record in reference_genome
        ) - 1,  # Adjusting for the self-match if necessary
        axis=1
    )
    # Reorganise the guide RNA dataframe
    guides = guides[
        [
            "PAM",
            "Distance from Amino Acid (bp)",
            "Full gRNA Sequence",
            "Position on + strand",
            "Strand",
            "G/C Content (%)",
            "off target count (loci)",
            "off target count (genome)",
            "Notes",
        ]
    ]
    guides = guides.reset_index(drop = True)
    
    return guides

In [7]:
# Tests for guide RNA generation for a specific amino acid
display(test_genomic_loci_extracted)
test_loci = test_genomic_loci_extracted['genomic_loci'].iloc[0]
print(test_loci)
test_guides = specific_function_guide_RNA_generation(50, test_loci, test_sequence_records)
display(test_guides)

,accession,name,taxid,sequence,gnCoordinate.genomicLocation.chromosome,gnCoordinate.genomicLocation.start,gnCoordinate.genomicLocation.end,gnCoordinate.genomicLocation.reverseStrand,gnCoordinate.genomicLocation.nucleotideId,gnCoordinate.genomicLocation.assemblyName,...,gnCoordinate.ensemblTranscriptId,gnCoordinate.ensemblTranslationId,exon_id,exon_proteinLocation.begin.position,exon_proteinLocation.end.position,exon_genomeLocation.begin.position,exon_genomeLocation.end.position,genomic_loci,genomic_loci_exon,translation_check
0,A0A075B6Q4,A0A075B6Q4_HUMAN,9606,MDIQKSENEDDSEWEDVDDEKGDSNDDYDSAGLLSDEDCMSVPGKT...,6,143857434,143884328,False,CM000668.2,GRCh38,...,ENST00000454207,ENSP00000400756,"ENSE00003758190,ENSE00003759740,ENSE0000375999...","1,4,90,132","4,89,132,140","143857434,143857752,143860426,143884304","143857444,143858007,143860553,143884328",gtctgtcccttagcagagcttgagcactgtgccgggagatccgctg...,ATGGATATACAGAAATCTGAGAATGAAGATGACAGCGAGTGGGAAG...,Pass


...
loci (length:27895)
...
loci exon (length:420)
ATGGATATACAGAAATCTGAGAATGAAGATGACAGCGAGTGGGAAGATGTGGATGATGAGAAGGGAGATAGCAATGATGACTATGACTCTGCAGGCCTATTGTCAGATGAAGACTGTATGTCTGTGCCCGGAAAAACTCACAGAGCTATAGCAGATCACTTGTTCTGGAGTGAGGAAACAAAGAGTCGCTTCACGGAGTATTCGATGACTTCCTCAGTCATGAGGAGAAATGAACAGCTGACCCTACATGATGAGAGGTTTGAGAAGTTTTATGAGCAATATGATGATGATGAAATTGGAGCTCTGGATAATGCAGAATTGGAAGGTTCTATTCAAGTGGACAGCAATCGCTTACAGGAAGTTTTGAATGACTACTATAAAGAGAAGGCAGAGAAATGGCAATCAGGAATTGTTTCCCTG
The amino acid you have selected is {'Amino Acid': 'I', 'base_1': (np.int64(954), 'A'), 'base_2': (np.int64(955), 'T'), 'base_3': (np.int64(956), 'A')} at aa position 50 and loci postion 954 - 956
The PAM you have selected is NGG
Searching for gRNAs in loci region 834-1076...
GATGACAGCGAGTGGGAAGATGTGGATGATGAGAAGGGAGATAGCAATGATGACTATGACTCTGCAGGCCTATTGTCAGATGAAGACTGTATGTCTGTGCCCGGAAAAACTCACAGAGCTATAGCAGATCACTTGTTCTGGAGTGAGGAAACAAAGAGTCGCTTCACGGAGTATTCGATGACTTCCTCAGTCATGAGGAGAAATGAACAGCTGACCCTACATGATGAGAGGTTTGAGAAGgt
Searchin

,PAM,Distance from Amino Acid (bp),Full gRNA Sequence,Position on + strand,Strand,G/C Content (%),off target count (loci),off target count (genome),Notes
0,AGG,-55,CAATGATGACTATGACTCTGC,899,forward,42.86,0,0,[No leading G.]
1,AGG,-49,ATACAGTCTTCATCTGACAAT,905,reverse,33.33,0,0,[No leading G.]
2,CGG,44,AGGAAACAAAGAGTCGCTTCA,1000,forward,42.86,0,0,[No leading G.]


In [8]:
#hdr template generation 
def select_guide_pair(specific_function_output):
    """
    Allows the user to select two rows from a specific function output
    and returns a new DataFrame with the selected rows.
    
    Arguments:
    specific_function_output: The guide RNA DataFrame to select rows from.
    
    Returns:
    upstream guide: row with upstream guide information 
    downstream_guide: row with downstream guide information 
    """
    # Display the DataFrame
    print("Here is the DataFrame of all guides:")
    print("Upstream guides have a negative distance and downstream guides have a positive distance")
    print("If possible, pick a reverse strand gRNA upstream guide and and forward strand downstream guide")
    print("This means that the guide RNA sequence will be recodonised")
    display(specific_function_output)
    
    # Allow the user to select two rows
    row1 = int(input(f"Enter the index of the upstream guide (distance to target aa < 0) [0-{len(specific_function_output)-1}]: "))
    row2 = int(input(f"Enter the index of the downstream guide (distance to target aa > 0) [0-{len(specific_function_output)-1}]: "))

    # Validate that the inputs are within the correct range
    if row1 not in specific_function_output.index or row2 not in specific_function_output.index:
        raise ValueError("Selected index is out of range.")

    # Fetch the selected rows and save them as a new DataFrame
    upstream_guide = specific_function_output.iloc[[row1]].reset_index(drop = True)
    downstream_guide = specific_function_output.iloc[[row2]].reset_index(drop = True)
    
    # Display the new DataFrame
    print("\nYou have selected the following rows:")
    display(upstream_guide)
    display(downstream_guide)

    print("Extracting sequences:")
    upstream_guide_sequence, upstream_guide_position, upstream_guide_distance = (
    upstream_guide[['Full gRNA Sequence', 'Position on + strand', 'Distance from Amino Acid (bp)']].iloc[0]
    )   
    downstream_guide_sequence, downstream_guide_position, downstream_guide_distance = (
    downstream_guide[['Full gRNA Sequence', 'Position on + strand', 'Distance from Amino Acid (bp)']].iloc[0]
    )
    
        
    return upstream_guide_sequence, upstream_guide_position, upstream_guide_distance, downstream_guide_sequence, downstream_guide_position, downstream_guide_distance

def generate_hdr_template(loci, upstream_guide_position, downstream_guide_position, homology_overlap = 100):
    """
    Finds the sequence between two guide RNA sequences in a genomic loci and adds an homology overlap.
    
    Arguments:
    loci (str): Genomic loci that the guides were generated from.
    upstream_guide_position: position of the start of the upstream guide PAM
    downstream_guide_position: position of the start of the downstream guide PAM
    homology_overlap (int): Number of nucleotides to pad around the sequence found between the guides. Default = 100

    Returns:
    hdr_template (str): Sequence found between the start of the PAM sites for the gRNA cut sites.
    hdr_template_plus_homology (str): Sequence found between the gRNAs, padded by homology overlap.
    selected_guides_distance_from_aa: distance from target amino acid for selected guides. Upstream guide is distance to start of the codon. Downstream guide is distance to end of the codon.
    """
     
    # Get the start and end indices in the loci of the region between guides
    start_hdr_template = int(upstream_guide_position)
    end_hdr_template = int(downstream_guide_position)

        
    # Calculate the indices for the start and end of homology arms
    upstream_homology_arm_start = max(0, start_hdr_template - homology_overlap)
    upstream_homology_arm_end = upstream_guide_position
    downstream_homology_arm_start = downstream_guide_position
    downstream_homology_arm_end = min(len(loci), end_hdr_template + homology_overlap)
        
    # Extract the homology arms
    hdr_template = loci[start_hdr_template:end_hdr_template] #plus one ensures that end hdr template position is included
    upstream_homology_arm = loci[upstream_homology_arm_start:upstream_homology_arm_end]
    downstream_homology_arm = loci[downstream_homology_arm_start:downstream_homology_arm_end+1]
    

    return hdr_template, upstream_homology_arm,upstream_homology_arm_start, upstream_homology_arm_end, downstream_homology_arm, downstream_homology_arm_start, downstream_homology_arm_end

def recodonise_sequence(hdr_template, upstream_guide_distance, downstream_guide_distance):
    """
    Recodonise dna sequence to most abundant codons but not changing protein sequence
    """
    # Dictionaries required for function
    codon_dict = {
    'GCT': 'A', 'GCC': 'A', 'GCA': 'A', 'GCG': 'A',
    'CGT': 'R', 'CGC': 'R', 'CGA': 'R', 'CGG': 'R', 'AGA': 'R', 'AGG': 'R',
    'AAT': 'N', 'AAC': 'N',
    'GAT': 'D', 'GAC': 'D',
    'TGT': 'C', 'TGC': 'C',
    'GAA': 'E', 'GAG': 'E',
    'CAA': 'Q', 'CAG': 'Q',
    'GGT': 'G', 'GGC': 'G', 'GGA': 'G', 'GGG': 'G',
    'CAT': 'H', 'CAC': 'H',
    'ATT': 'I', 'ATC': 'I', 'ATA': 'I',
    'TTA': 'L', 'TTG': 'L', 'CTT': 'L', 'CTC': 'L', 'CTA': 'L', 'CTG': 'L',
    'AAA': 'K', 'AAG': 'K',
    'ATG': 'M',  # Also the start codon
    'TTT': 'F', 'TTC': 'F',
    'CCT': 'P', 'CCC': 'P', 'CCA': 'P', 'CCG': 'P',
    'TCT': 'S', 'TCC': 'S', 'TCA': 'S', 'TCG': 'S', 'AGT': 'S', 'AGC': 'S',
    'ACT': 'T', 'ACC': 'T', 'ACA': 'T', 'ACG': 'T',
    'TGG': 'W',
    'TAT': 'Y', 'TAC': 'Y',
    'GTT': 'V', 'GTC': 'V', 'GTA': 'V', 'GTG': 'V',
    'TAA': '*', '*': '*', 'TGA': '*'
    }

    aa_to_codon = {
    'I': ['ATA', 'ATC', 'ATT'], # Isoleucine
    'M': ['ATG'], # Methionine
    'T': ['ACA', 'ACC', 'ACG', 'ACT'], #Threonine
    'N': ['AAC', 'AAT'], # Asparginine 
    'K': ['AAA', 'AAG'], #Lysine
    'S': ['AGC', 'AGT', 'TCA', 'TCC', 'TCG', 'TCT'],  # Serine
    'R': ['AGA', 'AGG', 'CGA', 'CGC', 'CGG', 'CGT'],  # Arginine
    'L': ['CTA', 'CTC', 'CTG', 'CTT', 'TTA', 'TTG'], # Leucine
    'P': ['CCA', 'CCC', 'CCG', 'CCT'], # Proline
    'H': ['CAC', 'CAT'], # Histidine 
    'Q': ['CAA', 'CAG'], # Gluatmine
    'V': ['GTA', 'GTC', 'GTG', 'GTT'], # Valine
    'A': ['GCA', 'GCC', 'GCG', 'GCT'], # Alanine
    'D': ['GAC', 'GAT'], # Aspartic Acid
    'E': ['GAA', 'GAG'], # Glutamic Acid 
    'G': ['GGA', 'GGC', 'GGG', 'GGT'], # Glycine
    'F': ['TTC', 'TTT'], #Phenylalanine
    'Y': ['TAC', 'TAT'], #Tyrosine
    'C': ['TGC', 'TGT'], #Cysteine
    'W': ['TGG'], #Tryptophan
    '*': ['TAA', 'TAG', 'TGA']  # Stop codons
    }

    human_frequency_dict = {
    # Isoleucine (Ile)
    'ATA': 0.16,
    'ATC': 0.48,
    'ATT': 0.36,
    
    # Methionine (Met)
    'ATG': 1.00,  
    
    # Threonine (Thr)
    'ACA': 0.28,
    'ACC': 0.36,
    'ACG': 0.12,
    'ACT': 0.24,
    
    # Asparagine (Asn)
    'AAC': 0.54,
    'AAT': 0.46,
    
    # Lysine (Lys)
    'AAA': 0.42,
    'AAG': 0.58,
    
    # Serine (Ser)
    'AGC': 0.24,
    'AGT': 0.15,
    
    # Arginine (Arg)
    'AGA': 0.20,
    'AGG': 0.20,
    
    # Leucine (Leu)
    'CTA': 0.07,
    'CTC': 0.20,
    'CTG': 0.41,
    'CTT': 0.13,
    
    # Proline (Pro)
    'CCA': 0.27,
    'CCC': 0.33,
    'CCG': 0.11,
    'CCT': 0.28,
    
    # Histidine (His)
    'CAC': 0.59,
    'CAT': 0.41,
    
    # Glutamine (Gln)
    'CAA': 0.25,
    'CAG': 0.75,
    
    # Arginine (Arg)
    'CGA': 0.11,
    'CGC': 0.19,
    'CGG': 0.21,
    'CGT': 0.08,
    
    # Valine (Val)
    'GTA': 0.11,
    'GTC': 0.24,
    'GTG': 0.47,
    'GTT': 0.18,
    
    # Alanine (Ala)
    'GCA': 0.23,
    'GCC': 0.40,
    'GCG': 0.11,
    'GCT': 0.26,
    
    # Aspartic acid (Asp)
    'GAC': 0.54,
    'GAT': 0.46,
    
    # Glutamic acid (Glu)
    'GAA': 0.42,
    'GAG': 0.58,
    
    # Glycine (Gly)
    'GGA': 0.25,
    'GGC': 0.34,
    'GGG': 0.25,
    'GGT': 0.16,
    
    # Serine (Ser)
    'TCA': 0.15,
    'TCC': 0.22,
    'TCG': 0.06,
    'TCT': 0.18,
    
    # Phenylalanine (Phe)
    'TTC': 0.55,
    'TTT': 0.45,
    
    # Leucine (Leu)
    'TTA': 0.07,
    'TTG': 0.13,
    
    # Tyrosine (Tyr)
    'TAC': 0.57,
    'TAT': 0.43,
    
    # Cysteine (Cys)
    'TGC': 0.55,
    'TGT': 0.45,
    
    # Tryptophan (Trp)
    'TGG': 1.00,
    
    # Stop (*)
    'TAA': 0.61,
    'TAG': 0.09,
    'TGA': 0.30
    }
    
    #Check in hdr_template_exon is in frame and shift start point of recodonisation to be in frame
    #this is done by identifying the target codon and frameshifting to that reference
    hdr_template, hdr_template_exon = clean_loci(hdr_template)
    spec_aa_codon_start = 0 + abs(upstream_guide_distance)
    spec_aa_codon_end = len(hdr_template) - downstream_guide_distance
    print(spec_aa_codon_end)
    spec_aa_codon = hdr_template[spec_aa_codon_start:spec_aa_codon_end+1]
    print(f'hdr template(length:{len(hdr_template)})')
    print(hdr_template)
    
    print(f'hdr template exon (length:{len(hdr_template_exon)})')
    print(hdr_template_exon)
    
    print(f'Target codon is at position: {spec_aa_codon_start} - {spec_aa_codon_end}')
    print(f'{spec_aa_codon}:{translate(spec_aa_codon)}')
    
    # Correct spec_aa_codon_start and end for possible introns
    exon_start_index = next((i for i, char in enumerate(hdr_template) if char.isupper()), -1)
    spec_aa_codon_start_exon_corrected = spec_aa_codon_start - exon_start_index

    if spec_aa_codon_start_exon_corrected % 3 == 0:
        print('Exons are in frame')
        n = 0
    elif spec_aa_codon_start_exon_corrected % 3 == 1:
        print('Exons are out of frame by 1 position. Shifting start of recodonisation by 1')
        n = 1
    elif spec_aa_codon_start_exon_corrected % 3 == 2:
        print('Exons are out of frame by 2 positions. Shifting start of recodonisation by 2')
        n = 2
        
    #changing codons to aa
    aa_sequence = ""
    hdr_template_exon_dict = {}
    exon_start_index = next((i for i, char in enumerate(hdr_template) if char.isupper()), -1)
    for i in range(n, len(hdr_template_exon)-3, 3):
        codon_start = i
        codon_end = i+3
        codon = hdr_template_exon[codon_start:codon_end]
        aa = codon_dict[codon]
        aa_sequence += aa
        hdr_template_exon_dict[i // 3] = codon
   
    print(f'hdr sequence to be recodonised(length:{len(hdr_template[n::])})')
    print(hdr_template[n::])
    print('aa sequence')
    print(aa_sequence)
    
    recodonised_sequence = ""
    
    
    for m, aa in enumerate(aa_sequence):
        current_codon = hdr_template_exon_dict[m]

        if aa in ['M', "W", "*"]:
            recodonised_sequence += current_codon
        else:
            aa_dict = {codon : human_frequency_dict[codon] for codon in aa_to_codon[aa]}
            codon_freq_list = sorted(aa_dict.items(), key=lambda x: x[1], reverse=True)
            for i, codon_freq in enumerate(codon_freq_list):
                if (codon_freq[0] == current_codon) and (i == 0):
                    recodonised_sequence += codon_freq_list[1][0]  # lower down a rank
                elif codon_freq[0] == current_codon:
                    recodonised_sequence += codon_freq_list[i-1][0]  # upper a rank
                    
    print(f'recodonised sequence (length:{len(recodonised_sequence)})')
    print(recodonised_sequence)
    
    print('translating recodonised sequence')
    recodonised_sequence_translation = translate(recodonised_sequence)
    print(recodonised_sequence_translation)
    
    if aa_sequence == recodonised_sequence_translation:
        print('Recodonised translations matches original translation')
    else:
        print('Recodonisation failed')
    
    # Now reattach the recodonised hdr region back to the hdr template
    # find region upstream to reattach to recodonised sequence. +n + 1 correct for the earlier frameshift correction
    exon_start_index = next((i for i, char in enumerate(hdr_template) if char.isupper()), -1)
    upstream_hdr_template = hdr_template[0:exon_start_index+n]
    print('Upstream hdr template')
    print(upstream_hdr_template)
    downstream_hdr_template = hdr_template[exon_start_index+n+len(recodonised_sequence)::]
    print('Downstream hdr template')
    print(downstream_hdr_template)
    
    hdr_template_recodonised = upstream_hdr_template + recodonised_sequence + downstream_hdr_template
    print(f'recodonised hdr template(length:{len(hdr_template_recodonised)})')
    print(hdr_template_recodonised)
    
    if len(hdr_template) == len(hdr_template_recodonised):
        print('Length of hdr template = length of recodonised hdr template')
        recodonisation_check = 'Pass'
    else:
        print('hdr template reconstruction failed')
        recodonisation_check = 'Fail'
        
    # return the recodonized sequence
    return hdr_template_recodonised, spec_aa_codon_start, recodonisation_check

In [12]:
# test hdr generation
test_upstream_guide_sequence, test_upstream_guide_position, test_upstream_guide_distance, test_downstream_guide_sequence, test_downstream_guide_position, test_downstream_guide_distance = select_guide_pair(test_guides)

test_hdr_template, test_upstream_homology_arm, test_upstream_homology_arm_start_postion, test_upstream_homology_arm_end_position, test_downstream_homology_arm, test_downstream_homology_arm_start_position, test_downstream_homology_arm_end_position = generate_hdr_template(test_loci, test_upstream_guide_position, test_downstream_guide_position, homology_overlap = 100)
print(test_hdr_template)

test_hdr_template_recodonised, test_spec_aa_codon_start, test_recodonisation_check = recodonise_sequence(test_hdr_template, test_upstream_guide_distance, test_downstream_guide_distance)
print(test_hdr_template_recodonised)


Here is the DataFrame of all guides:
Upstream guides have a negative distance and downstream guides have a positive distance
If possible, pick a reverse strand gRNA upstream guide and and forward strand downstream guide
This means that the guide RNA sequence will be recodonised


,PAM,Distance from Amino Acid (bp),Full gRNA Sequence,Position on + strand,Strand,G/C Content (%),off target count (loci),off target count (genome),Notes
0,AGG,-55,CAATGATGACTATGACTCTGC,899,forward,42.86,0,0,[No leading G.]
1,AGG,-49,ATACAGTCTTCATCTGACAAT,905,reverse,33.33,0,0,[No leading G.]
2,CGG,44,AGGAAACAAAGAGTCGCTTCA,1000,forward,42.86,0,0,[No leading G.]



You have selected the following rows:


,PAM,Distance from Amino Acid (bp),Full gRNA Sequence,Position on + strand,Strand,G/C Content (%),off target count (loci),off target count (genome),Notes
0,AGG,-49,ATACAGTCTTCATCTGACAAT,905,reverse,33.33,0,0,[No leading G.]


,PAM,Distance from Amino Acid (bp),Full gRNA Sequence,Position on + strand,Strand,G/C Content (%),off target count (loci),off target count (genome),Notes
0,CGG,44,AGGAAACAAAGAGTCGCTTCA,1000,forward,42.86,0,0,[No leading G.]


Extracting sequences:
ATTGTCAGATGAAGACTGTATGTCTGTGCCCGGAAAAACTCACAGAGCTATAGCAGATCACTTGTTCTGGAGTGAGGAAACAAAGAGTCGCTTCA
51
hdr template(length:95)
ATTGTCAGATGAAGACTGTATGTCTGTGCCCGGAAAAACTCACAGAGCTATAGCAGATCACTTGTTCTGGAGTGAGGAAACAAAGAGTCGCTTCA
hdr template exon (length:95)
ATTGTCAGATGAAGACTGTATGTCTGTGCCCGGAAAAACTCACAGAGCTATAGCAGATCACTTGTTCTGGAGTGAGGAAACAAAGAGTCGCTTCA
Target codon is at position: 49 - 51
ATA:I
Exons are out of frame by 1 position. Shifting start of recodonisation by 1
hdr sequence to be recodonised(length:94)
TTGTCAGATGAAGACTGTATGTCTGTGCCCGGAAAAACTCACAGAGCTATAGCAGATCACTTGTTCTGGAGTGAGGAAACAAAGAGTCGCTTCA
aa sequence
LSDEDCMSVPGKTHRAIADHLFWSEETKSRF
recodonised sequence (length:93)
CTTAGTGACGAGGATTGCATGTCCGTCCCTGGCAAGACACATCGGGCCATTGCTGACCATCTTTTTTGGTCTGAAGAGACCAAATCTAGGTTT
translating recodonised sequence
LSDEDCMSVPGKTHRAIADHLFWSEETKSRF
Recodonised translations matches original translation
Upstream hdr template
A
Downstream hdr template
A
recodonised hdr template(length:95)
A

In [13]:
#primer generation for amplicons
# Design primers for amplicon library generation using primer3 
# First create a function to identify positions of the start and end of the homology arms in the loci
import primer3

def generate_integration_specific_primers(loci, hdr_template_recodonised, upstream_homology_arm_end_position, downstream_homology_arm_start_position):
    # Construct the recodonised loci
    upstream_loci = loci[0:upstream_homology_arm_end_position]
    downstream_loci = loci[downstream_homology_arm_start_position::]
    loci_plus_recodonised_hdr_template = upstream_loci + hdr_template_recodonised + downstream_loci
    if len(loci) == len(loci_plus_recodonised_hdr_template):
        print('Reconstruction successful')
        recodonised_loci_reconstruction_check = 'Pass'
    else:
        print('Reconstruction failed')
        recodonised_loci_reconstruction_check = 'Fail'

    # Run primer3 workflow
    seq_args = {} 
    seq_args['SEQUENCE_TEMPLATE'] = loci
    seq_args['SEQUENCE_FORCE_LEFT_END'] = upstream_homology_arm_end_position + 20

    try:    
        primers = primer3.bindings.design_primers(
        seq_args= seq_args,
        global_args={
            'PRIMER_OPT_SIZE': 20,
            'PRIMER_MIN_SIZE': 15,
            'PRIMER_MAX_SIZE': 25,
            'PRIMER_OPT_TM': 60.0,
            'PRIMER_MIN_TM': 58.0,
            'PRIMER_MAX_TM': 62.0,
            'PRIMER_MIN_GC': 20.0,
            'PRIMER_MAX_GC': 80.0,
            'PRIMER_SALT_MONOVALENT': 50.0,
            'PRIMER_DNA_CONC': 500.0,
            'PRIMER_PRODUCT_SIZE_RANGE': [
                [400,500]
            ],
        })
        print(primers)
        # If no primers are returned, raise an exception
        if primers['PRIMER_LEFT_NUM_RETURNED'] == 0:
            raise ValueError("No primers found")
        
        left_primer_sequence = primers['PRIMER_LEFT'][0]['SEQUENCE']
        left_primer_tm = primers['PRIMER_LEFT'][0]['TM']
        right_primer_sequence = primers['PRIMER_RIGHT'][0]['SEQUENCE']
        right_primer_tm = primers['PRIMER_RIGHT'][0]['TM']
        return(loci_plus_recodonised_hdr_template, recodonised_loci_reconstruction_check, left_primer_sequence, left_primer_tm, right_primer_sequence, right_primer_tm)
    
    except ValueError as e:
        print(e)
        # Handle the case where no primers are found and continue by making overlap primer on the right side
        seq_args = {} 
        seq_args['SEQUENCE_TEMPLATE'] = loci
        seq_args['SEQUENCE_FORCE_RIGHT_END'] = downstream_homology_arm_start_position - 20
        primers = primer3.bindings.design_primers(
        seq_args= seq_args,
        global_args={
            'PRIMER_OPT_SIZE': 20,
            'PRIMER_MIN_SIZE': 15,
            'PRIMER_MAX_SIZE': 25,
            'PRIMER_OPT_TM': 60.0,
            'PRIMER_MIN_TM': 58.0,
            'PRIMER_MAX_TM': 62.0,
            'PRIMER_MIN_GC': 20.0,
            'PRIMER_MAX_GC': 80.0,
            'PRIMER_SALT_MONOVALENT': 50.0,
            'PRIMER_DNA_CONC': 500.0,
            'PRIMER_PRODUCT_SIZE_RANGE': [
                [400,500]
            ],
        })
        print(primers)
        left_primer_sequence = primers['PRIMER_LEFT'][0]['SEQUENCE']
        left_primer_tm = primers['PRIMER_LEFT'][0]['TM']
        right_primer_sequence = primers['PRIMER_RIGHT'][0]['SEQUENCE']
        right_primer_tm = primers['PRIMER_RIGHT'][0]['TM']
        return(loci_plus_recodonised_hdr_template, recodonised_loci_reconstruction_check, left_primer_sequence, left_primer_tm, right_primer_sequence, right_primer_tm)


In [14]:
#test generate integration specific primers
test_hdr_template_recodonised
test_spec_aa_codon_start 
test_recodonisation_check

test_loci_plus_recodonised_hdr_template, test_recodonised_loci_reconstruction_check, test_left_primer_sequence, test_left_primer_tm, test_right_primer_sequence, test_right_primer_tm = generate_integration_specific_primers(test_loci, test_hdr_template_recodonised, test_upstream_homology_arm_end_position, test_downstream_homology_arm_start_position)

Reconstruction successful
{'PRIMER_LEFT_EXPLAIN': 'considered 11, low tm 9, ok 2', 'PRIMER_RIGHT_EXPLAIN': 'considered 295415, GC content failed 15468, low tm 154368, high tm 66143, high hairpin stability 4, long poly-x seq 1088, ok 58344', 'PRIMER_PAIR_EXPLAIN': 'considered 3121, unacceptable product size 3116, ok 5', 'PRIMER_LEFT_NUM_RETURNED': 5, 'PRIMER_RIGHT_NUM_RETURNED': 5, 'PRIMER_INTERNAL_NUM_RETURNED': 0, 'PRIMER_PAIR_NUM_RETURNED': 5, 'PRIMER_PAIR': [{'PENALTY': 5.626187951312033, 'COMPL_ANY_TH': 0.0, 'COMPL_END_TH': 0.0, 'PRODUCT_SIZE': 489, 'PRODUCT_TM': 84.92367148841798}, {'PENALTY': 5.626645138941285, 'COMPL_ANY_TH': 0.0, 'COMPL_END_TH': 0.0, 'PRODUCT_SIZE': 455, 'PRODUCT_TM': 84.62209617211875}, {'PENALTY': 5.752228979977929, 'COMPL_ANY_TH': 0.0, 'COMPL_END_TH': 0.0, 'PRODUCT_SIZE': 500, 'PRODUCT_TM': 84.97145880948138}, {'PENALTY': 6.050121615892067, 'COMPL_ANY_TH': 13.208390410764025, 'COMPL_END_TH': 2.9169102374171416, 'PRODUCT_SIZE': 432, 'PRODUCT_TM': 84.628440290

In [13]:
import pandas as pd 
codon_usage_table_file = 'codon_usage_human.xlsx'
codon_usage_table = pd.read_excel(codon_usage_table_file)
display(codon_usage_table)
codon_usage_table_dict = dict(zip(codon_usage_table['Codon'], codon_usage_table['Frequency']))

print(codon_usage_table_dict)

#now sort the inital codon usage table and keep only the highest used codon for each amino acid
df_sorted = codon_usage_table.sort_values(['Amino_Acid', 'Frequency'], ascending=[True, False])
# Keep the first row for each AminoAcid (highest frequency)
top_codon = df_sorted.groupby('Amino_Acid', as_index=False).first()
display(top_codon)
top_codon_list = list(zip(top_codon['Amino_Acid'], top_codon['Codon']))
print(top_codon_list)

,Amino_Acid,Codon,Frequency
0,G,GGG,669768
1,G,GGA,669873
2,G,GGT,437126
3,G,GGC,903565
4,E,GAG,1609975
...,...,...,...
59,L,CTC,796638
60,P,CCG,281570
61,P,CCA,688038
62,P,CCT,713233


{'GGG': 669768, 'GGA': 669873, 'GGT': 437126, 'GGC': 903565, 'GAG': 1609975, 'GAA': 1177632, 'GAT': 885429, 'GAC': 1020595, 'GTG': 1143534, 'GTA': 287712, 'GTT': 448607, 'GTC': 588138, 'GCG': 299495, 'GCA': 643471, 'GCT': 750096, 'GCC': 1127679, 'AGG': 486463, 'AGA': 494682, 'AGT': 493429, 'AGC': 791383, 'AAG': 1295568, 'AAA': 993621, 'AAT': 689701, 'AAC': 776603, 'ATG': 896005, 'ATA': 304565, 'ATT': 650473, 'ATC': 846466, 'ACG': 246105, 'ACA': 614523, 'ACT': 533609, 'ACC': 768147, 'TGG': 535595, 'TGA': 63237, 'TGT': 430311, 'TGC': 513028, 'TAG': 32109, 'TAA': 40285, 'TAT': 495699, 'TAC': 622407, 'TTG': 525688, 'TTA': 311881, 'TTT': 714298, 'TTC': 824692, 'TCG': 179419, 'TCA': 496448, 'TCT': 618711, 'TCC': 718892, 'CGG': 464485, 'CGA': 250760, 'CGT': 184609, 'CGC': 423516, 'CAG': 1391973, 'CAA': 501911, 'CAT': 441711, 'CAC': 613713, 'CTG': 1611801, 'CTA': 290751, 'CTT': 536515, 'CTC': 796638, 'CCG': 281570, 'CCA': 688038, 'CCT': 713233, 'CCC': 804620}


,Amino_Acid,Codon,Frequency
0,*,TGA,63237
1,A,GCC,1127679
2,C,TGC,513028
3,D,GAC,1020595
4,E,GAG,1609975
5,F,TTC,824692
6,G,GGC,903565
7,H,CAC,613713
8,I,ATC,846466
9,K,AAG,1295568


[('*', 'TGA'), ('A', 'GCC'), ('C', 'TGC'), ('D', 'GAC'), ('E', 'GAG'), ('F', 'TTC'), ('G', 'GGC'), ('H', 'CAC'), ('I', 'ATC'), ('K', 'AAG'), ('L', 'CTG'), ('M', 'ATG'), ('N', 'AAC'), ('P', 'CCC'), ('Q', 'CAG'), ('R', 'AGA'), ('S', 'AGC'), ('T', 'ACC'), ('V', 'GTG'), ('W', 'TGG'), ('Y', 'TAC')]
